# 03 · Baseline Models

Build metadata + text features and train interpretable baselines (Logistic Regression, Random Forest) for `is_risky` (**RQ1**).

- **Inputs:** `data/sample/sample_prs.csv`
- **Outputs:** Trained baselines, an evaluation table, a confusion matrix, and a feature-set comparison.

> ⚠️ **Sample vs. real data.** This notebook runs on the committed 10-row synthetic sample so the toolchain works without PRismBench. The sample has singleton classes, so metrics here are *illustrative only*. Each `TODO` marks where the real dataset in `data/raw/` plugs in.

In [ ]:
# --- Standard setup: locate project root, add src/ to path, load helpers ---
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the repo root (has pyproject.toml + src/pr_risk)."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "pr_risk").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 50)
SAMPLE_CSV = PROJECT_ROOT / "data" / "sample" / "sample_prs.csv"
print("Project root :", PROJECT_ROOT)
print("Sample CSV   :", SAMPLE_CSV.name, "| exists:", SAMPLE_CSV.exists())

In [ ]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid")
    HAS_SNS = True
except ImportError:  # seaborn is optional; matplotlib is enough
    HAS_SNS = False
print("seaborn available:", HAS_SNS)

## 1. Load & prepare
Use the binary target `is_risky ∈ {0, 1}` (the runnable target on the sample).

In [ ]:
from pr_risk.data.load_data import load_csv

df = load_csv(SAMPLE_CSV)
df = df[df["is_risky"].isin([0, 1])].copy()
df["is_risky"] = df["is_risky"].astype(int)
df["text"] = (df["title"].fillna("") + " " + df["description"].fillna("")).str.lower()
print("rows:", len(df), "| class balance:", df["is_risky"].value_counts().to_dict())

## 2. Split
Simple non-stratified split for the tiny sample (see nb 02 for the stratified real-data path).

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
print("train:", len(train_df), "test:", len(test_df))

## 3. Feature engineering
Metadata features + TF-IDF over text, combined into one sparse matrix.

In [ ]:
from pr_risk.features.feature_pipeline import combine_features
from pr_risk.features.metadata_features import create_metadata_features
from pr_risk.features.text_features import create_text_features_tfidf

meta_train = create_metadata_features(train_df)
meta_test = create_metadata_features(test_df)

text_tr, _text_val, text_te, vectorizer = create_text_features_tfidf(
    train_df["text"], train_df["text"], test_df["text"], max_features=200
)

X_train = combine_features(meta_train, text_tr)
X_test = combine_features(meta_test, text_te)
y_train, y_test = train_df["is_risky"], test_df["is_risky"]

feature_names = list(meta_train.columns) + list(vectorizer.get_feature_names_out())
print("combined feature matrix:", X_train.shape, "->", len(feature_names), "features")

## 4. Train baselines

In [ ]:
from pr_risk.models.train_baseline import train_logistic_regression, train_random_forest

lr = train_logistic_regression(X_train, y_train)
rf = train_random_forest(X_train, y_train)
print("trained:", type(lr).__name__, "and", type(rf).__name__)

## 5. Evaluate

In [ ]:
from pr_risk.models.evaluate import evaluate_classification_model

rows = []
for name, model in [("logistic_regression", lr), ("random_forest", rf)]:
    res = evaluate_classification_model(model, X_test, y_test)
    scores = {k: round(v, 3) for k, v in res.items() if k != "classification_report"}
    rows.append({"model": name, **scores})

results = pd.DataFrame(rows).set_index("model")
display(results)
print(evaluate_classification_model(rf, X_test, y_test)["classification_report"])

## 6. Confusion matrix (Random Forest)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, cmap="Blues")
plt.title("Random Forest · is_risky")
plt.tight_layout()
plt.show()

## 7. Feature-set comparison
Compare metadata-only vs text-only vs combined (the plan from `docs/experiment_plan.md`).

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

feature_sets = {
    "metadata_only": (meta_train, meta_test),
    "text_only": (text_tr, text_te),
    "combined": (combine_features(meta_train, text_tr), combine_features(meta_test, text_te)),
}

comparison = []
for name, (Xtr, Xte) in feature_sets.items():
    model = train_random_forest(Xtr, y_train)
    preds = model.predict(Xte)
    comparison.append({
        "feature_set": name,
        "accuracy": round(accuracy_score(y_test, preds), 3),
        "f1_weighted": round(f1_score(y_test, preds, average="weighted", zero_division=0), 3),
    })
comparison_df = pd.DataFrame(comparison).set_index("feature_set")
display(comparison_df)
comparison_df.plot(kind="bar", figsize=(6, 3), title="Feature-set comparison (RF, sample)")
plt.tight_layout()
plt.show()

## 8. `risk_type` (illustrative)
Multi-class needs more data per class than the sample provides — shown here only as mechanics.

In [ ]:
full = load_csv(SAMPLE_CSV)
display(full["risk_type"].value_counts().to_frame("count"))
print("TODO: train a multi-class risk_type model on the real dataset (see configs/codebert.yaml).")

## Next steps / TODO (real data)
- Add XGBoost/LightGBM baselines (configs already exist).
- Tune TF-IDF (`max_features`, `ngram_range`) and add `code_features`.
- Train and evaluate the multi-class `risk_type` classifier.
- Log results to `experiments/experiment_log.csv`; carry strong models into **05** and **06**.